# Safety Data Processing

Load the local HarmMetric Eval benchmark dataset and the official local HarmBench text/multimodal behavior datasets.


In [ ]:
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict, get_dataset_config_names, get_dataset_split_names, load_dataset

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / ".git").exists():
    repo_root = repo_root.parent

repo_root


## Paths


In [ ]:
HARMMETRIC_DATASET_PATH = repo_root / "HarmMetric_Eval" / "data" / "dataset.jsonl"
HARMBENCH_TEXT_ALL_PATH = repo_root / "HarmBench" / "data" / "behavior_datasets" / "harmbench_behaviors_text_all.csv"
HARMBENCH_MULTIMODAL_ALL_PATH = repo_root / "HarmBench" / "data" / "behavior_datasets" / "harmbench_behaviors_multimodal_all.csv"

for path in [HARMMETRIC_DATASET_PATH, HARMBENCH_TEXT_ALL_PATH, HARMBENCH_MULTIMODAL_ALL_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)
    print(path)


## Load Data


In [ ]:
harmmetric_eval_df = pd.read_json(HARMMETRIC_DATASET_PATH, lines=True)
harmbench_text_all_df = pd.read_csv(HARMBENCH_TEXT_ALL_PATH)
harmbench_multimodal_all_df = pd.read_csv(HARMBENCH_MULTIMODAL_ALL_PATH)

print(f"HarmMetric Eval dataset: {harmmetric_eval_df.shape}")
print(f"HarmBench text_all: {harmbench_text_all_df.shape}")
print(f"HarmBench multimodal_all: {harmbench_multimodal_all_df.shape}")


## Column Overview


In [ ]:
dataset_overview_df = pd.DataFrame([
    {
        "name": "harmmetric_eval_df",
        "rows": len(harmmetric_eval_df),
        "columns": list(harmmetric_eval_df.columns),
    },
    {
        "name": "harmbench_text_all_df",
        "rows": len(harmbench_text_all_df),
        "columns": list(harmbench_text_all_df.columns),
    },
    {
        "name": "harmbench_multimodal_all_df",
        "rows": len(harmbench_multimodal_all_df),
        "columns": list(harmbench_multimodal_all_df.columns),
    },
])

dataset_overview_df


## Quick Counts


In [ ]:
print("HarmMetric Eval response categories")
display(harmmetric_eval_df["response_category"].value_counts(dropna=False).rename_axis("response_category").reset_index(name="count"))

print("HarmMetric Eval sources")
display(harmmetric_eval_df["source"].value_counts(dropna=False).rename_axis("source").reset_index(name="count"))

print("HarmBench text semantic categories")
display(harmbench_text_all_df["SemanticCategory"].value_counts(dropna=False).rename_axis("SemanticCategory").reset_index(name="count"))

print("HarmBench multimodal semantic categories")
display(harmbench_multimodal_all_df["SemanticCategory"].value_counts(dropna=False).rename_axis("SemanticCategory").reset_index(name="count"))


## Preview Metadata


In [ ]:
# Metadata-oriented previews. Use the dataframe variables directly if you want to inspect full prompt/response text.
display(harmmetric_eval_df[["prompt_id", "response_id", "response_category", "attack_method", "source"]].head())
display(harmbench_text_all_df[["BehaviorID", "FunctionalCategory", "SemanticCategory", "Tags"]].head())
display(harmbench_multimodal_all_df[["BehaviorID", "FunctionalCategory", "SemanticCategory", "ImageFileName", "Tags"]].head())


## Additional Safety Datasets From Hugging Face


In [ ]:
HF_SAFETY_DATASET_IDS = {
    "strongreject": "walledai/StrongREJECT",
    "malicious_instruct": "walledai/MaliciousInstruct",
    "advbench": "walledai/AdvBench",
    "jailbreakbench": "walledai/JailbreakBench",
}

HF_SAFETY_DATASET_IDS


In [ ]:
def load_hf_dataset_as_dataframes(dataset_id):
    """Load all configs/splits for a Hugging Face dataset into named DataFrames."""
    frames = {}
    try:
        config_names = get_dataset_config_names(dataset_id)
    except Exception:
        config_names = [None]

    for config_name in config_names:
        try:
            split_names = get_dataset_split_names(dataset_id, config_name) if config_name else get_dataset_split_names(dataset_id)
        except Exception:
            split_names = ["train"]

        for split_name in split_names:
            if config_name:
                dataset = load_dataset(dataset_id, config_name, split=split_name)
                key = f"{config_name}/{split_name}"
            else:
                dataset = load_dataset(dataset_id, split=split_name)
                key = split_name
            frames[key] = dataset.to_pandas()
            print(f"{dataset_id} [{key}]: {frames[key].shape}")
    return frames


hf_safety_df_by_dataset = {
    name: load_hf_dataset_as_dataframes(dataset_id)
    for name, dataset_id in HF_SAFETY_DATASET_IDS.items()
}


In [ ]:
strongreject_df = hf_safety_df_by_dataset["strongreject"][next(iter(hf_safety_df_by_dataset["strongreject"]))]
malicious_instruct_df = hf_safety_df_by_dataset["malicious_instruct"][next(iter(hf_safety_df_by_dataset["malicious_instruct"]))]
advbench_df = hf_safety_df_by_dataset["advbench"][next(iter(hf_safety_df_by_dataset["advbench"]))]
jailbreakbench_df = hf_safety_df_by_dataset["jailbreakbench"][next(iter(hf_safety_df_by_dataset["jailbreakbench"]))]

hf_safety_overview_df = pd.DataFrame([
    {
        "name": name,
        "dataset_id": HF_SAFETY_DATASET_IDS[name],
        "config_or_split": key,
        "rows": len(df),
        "columns": list(df.columns),
    }
    for name, frames in hf_safety_df_by_dataset.items()
    for key, df in frames.items()
])

hf_safety_overview_df


In [ ]:
# Metadata previews. Inspect each dataframe directly if you need full prompt text.
for name, df in [
    ("strongreject_df", strongreject_df),
    ("malicious_instruct_df", malicious_instruct_df),
    ("advbench_df", advbench_df),
    ("jailbreakbench_df", jailbreakbench_df),
]:
    print(f"{name}: {df.shape}")
    display(df.head())


## HarmMetric Eval Prompt Overlap With Source Benchmarks


In [ ]:
def _prompt_key(series):
    return pd.Series(series).dropna().astype(str).str.strip()


def exact_prompt_overlap(left_series, right_series):
    left = _prompt_key(left_series)
    right = _prompt_key(right_series)
    left_set = set(left)
    right_set = set(right)
    overlap = left_set & right_set
    return {
        "left_rows": len(left),
        "left_unique": len(left_set),
        "right_rows": len(right),
        "right_unique": len(right_set),
        "exact_overlap_unique": len(overlap),
        "left_unique_coverage": len(overlap) / len(left_set) if left_set else 0,
        "right_unique_coverage": len(overlap) / len(right_set) if right_set else 0,
        "overlap_prompts": sorted(overlap),
    }


harmmetric_prompts = harmmetric_eval_df["harmful_prompt"]
benchmark_prompt_sources = {
    "HarmBench_text_all": harmbench_text_all_df["Behavior"],
    "StrongREJECT": strongreject_df["prompt"],
    "MaliciousInstruct": malicious_instruct_df["prompt"],
    "AdvBench": advbench_df["prompt"],
    "JailbreakBench": jailbreakbench_df["prompt"],
}

overlap_records = []
overlap_detail_records = []
for benchmark_name, prompts in benchmark_prompt_sources.items():
    result = exact_prompt_overlap(harmmetric_prompts, prompts)
    overlap_records.append({
        "benchmark": benchmark_name,
        "harmmetric_rows": result["left_rows"],
        "harmmetric_unique_prompts": result["left_unique"],
        "benchmark_rows": result["right_rows"],
        "benchmark_unique_prompts": result["right_unique"],
        "exact_overlap_unique": result["exact_overlap_unique"],
        "harmmetric_unique_coverage": result["left_unique_coverage"],
        "benchmark_unique_coverage": result["right_unique_coverage"],
    })
    for prompt in result["overlap_prompts"]:
        overlap_detail_records.append({
            "benchmark": benchmark_name,
            "prompt": prompt,
        })

harmmetric_benchmark_overlap_df = pd.DataFrame(overlap_records)
harmmetric_benchmark_overlap_details_df = pd.DataFrame(overlap_detail_records)

harmmetric_benchmark_overlap_df


In [ ]:
SAFETY_OUTPUT_DIR = repo_root / "benchmarks" / "safety"
SAFETY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HARM_METRIC_OVERLAP_SUMMARY_PATH = SAFETY_OUTPUT_DIR / "harmmetric_prompt_overlap_summary.csv"
HARM_METRIC_OVERLAP_DETAILS_PATH = SAFETY_OUTPUT_DIR / "harmmetric_prompt_overlap_details.csv"

harmmetric_benchmark_overlap_df.to_csv(HARM_METRIC_OVERLAP_SUMMARY_PATH, index=False)
harmmetric_benchmark_overlap_details_df.to_csv(HARM_METRIC_OVERLAP_DETAILS_PATH, index=False)

print(f"Saved summary to {HARM_METRIC_OVERLAP_SUMMARY_PATH}")
print(f"Saved details to {HARM_METRIC_OVERLAP_DETAILS_PATH}")
print(f"Total exact overlap detail rows: {len(harmmetric_benchmark_overlap_details_df)}")


## Why Exact Overlap Can Exceed `source` Counts


In [ ]:
# HarmMetric Eval assigns one provenance label in `source`, but the same prompt string
# can appear in multiple benchmark datasets. This cell shows both views.

harmmetric_unique_prompt_source_df = (
    harmmetric_eval_df[["harmful_prompt", "source"]]
    .drop_duplicates()
    .groupby("source")["harmful_prompt"]
    .nunique()
    .sort_values(ascending=False)
    .rename("unique_prompts_by_harmmetric_source")
    .reset_index()
)

def exact_overlap_details_with_harmmetric_source(benchmark_name, benchmark_prompts):
    benchmark_prompt_set = set(_prompt_key(benchmark_prompts))
    matched = harmmetric_eval_df[["harmful_prompt", "source"]].drop_duplicates().copy()
    matched["prompt_match_key"] = _prompt_key(matched["harmful_prompt"])
    matched = matched.loc[matched["prompt_match_key"].isin(benchmark_prompt_set)].copy()
    matched["benchmark"] = benchmark_name
    return matched[["benchmark", "source", "harmful_prompt", "prompt_match_key"]]

source_overlap_detail_df = pd.concat(
    [
        exact_overlap_details_with_harmmetric_source("HarmBench_text_all", harmbench_text_all_df["Behavior"]),
        exact_overlap_details_with_harmmetric_source("StrongREJECT", strongreject_df["prompt"]),
        exact_overlap_details_with_harmmetric_source("MaliciousInstruct", malicious_instruct_df["prompt"]),
        exact_overlap_details_with_harmmetric_source("AdvBench", advbench_df["prompt"]),
        exact_overlap_details_with_harmmetric_source("JailbreakBench", jailbreakbench_df["prompt"]),
    ],
    ignore_index=True,
)

source_overlap_breakdown_df = (
    source_overlap_detail_df
    .groupby(["benchmark", "source"])
    .size()
    .rename("exact_prompt_overlap_count")
    .reset_index()
    .sort_values(["benchmark", "exact_prompt_overlap_count"], ascending=[True, False])
)

benchmark_overlap_totals_df = (
    source_overlap_detail_df
    .groupby("benchmark")["harmful_prompt"]
    .nunique()
    .rename("exact_overlap_unique")
    .reset_index()
    .sort_values("exact_overlap_unique", ascending=False)
)

print("Unique HarmMetric prompts by HarmMetric `source` tag")
display(harmmetric_unique_prompt_source_df)

print("Exact overlap totals by external benchmark")
display(benchmark_overlap_totals_df)

print("Where those exact-overlap prompts came from inside HarmMetric Eval")
display(source_overlap_breakdown_df)

# Use this dataframe to inspect the actual matching prompt strings.
source_overlap_detail_df


In [ ]:
SOURCE_OVERLAP_BREAKDOWN_PATH = SAFETY_OUTPUT_DIR / "harmmetric_prompt_overlap_by_harmmetric_source.csv"
SOURCE_OVERLAP_DETAIL_PATH = SAFETY_OUTPUT_DIR / "harmmetric_prompt_overlap_source_details.csv"

source_overlap_breakdown_df.to_csv(SOURCE_OVERLAP_BREAKDOWN_PATH, index=False)
source_overlap_detail_df.to_csv(SOURCE_OVERLAP_DETAIL_PATH, index=False)

print(f"Saved breakdown to {SOURCE_OVERLAP_BREAKDOWN_PATH}")
print(f"Saved source-level details to {SOURCE_OVERLAP_DETAIL_PATH}")


## Build SafetySuite JSON


In [ ]:
import json
from datetime import datetime, timezone

import numpy as np


def _jsonable(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonable(v) for v in value]
    return value


def _row_dict(row):
    return {str(k): _jsonable(v) for k, v in row.items()}


def _prompt_key(prompt):
    return str(prompt).strip()


def _ensure_suite_item(items_by_prompt, prompt):
    key = _prompt_key(prompt)
    if key not in items_by_prompt:
        items_by_prompt[key] = {
            "suite_id": f"safety_{len(items_by_prompt):05d}",
            "prompt": key,
            "source_datasets": [],
            "source_records": [],
        }
    return items_by_prompt[key]


def _add_suite_source_record(
    items_by_prompt,
    *,
    prompt,
    source_dataset,
    source_config="",
    source_split="",
    source_item_id="",
    modality="text",
    category=None,
    raw_record=None,
):
    item = _ensure_suite_item(items_by_prompt, prompt)
    if source_dataset not in item["source_datasets"]:
        item["source_datasets"].append(source_dataset)
    item["source_records"].append({
        "source_dataset": source_dataset,
        "source_config": source_config,
        "source_split": source_split,
        "source_item_id": str(_jsonable(source_item_id)) if source_item_id is not None else "",
        "modality": modality,
        "category": _jsonable(category),
        "raw_record": _jsonable(raw_record or {}),
    })


def build_safetysuite():
    items_by_prompt = {}

    for prompt, group in harmmetric_eval_df.groupby(harmmetric_eval_df["harmful_prompt"].map(_prompt_key), sort=False):
        first = group.iloc[0]
        response_records = [_row_dict(row) for _, row in group.iterrows()]
        _add_suite_source_record(
            items_by_prompt,
            prompt=prompt,
            source_dataset="HarmMetric_Eval",
            source_config="dataset",
            source_split="all",
            source_item_id=first.get("prompt_id", ""),
            modality="text",
            category=first.get("source", ""),
            raw_record={
                "prompt_id": first.get("prompt_id", ""),
                "source": first.get("source", ""),
                "num_response_records": len(response_records),
                "response_categories": sorted(set(str(x) for x in group["response_category"])),
                "attack_methods": sorted(set(str(x) for x in group["attack_method"])),
                "response_records": response_records,
            },
        )

    for _, row in harmbench_text_all_df.iterrows():
        _add_suite_source_record(
            items_by_prompt,
            prompt=row["Behavior"],
            source_dataset="HarmBench",
            source_config="text_all",
            source_split="all",
            source_item_id=row.get("BehaviorID", ""),
            modality="text",
            category=row.get("SemanticCategory", ""),
            raw_record=_row_dict(row),
        )

    for _, row in harmbench_multimodal_all_df.iterrows():
        _add_suite_source_record(
            items_by_prompt,
            prompt=row["Behavior"],
            source_dataset="HarmBench",
            source_config="multimodal_all",
            source_split="all",
            source_item_id=row.get("BehaviorID", ""),
            modality="multimodal",
            category=row.get("SemanticCategory", ""),
            raw_record=_row_dict(row),
        )

    hf_frames = [
        ("StrongREJECT", strongreject_df),
        ("MaliciousInstruct", malicious_instruct_df),
        ("AdvBench", advbench_df),
        ("JailbreakBench", jailbreakbench_df),
    ]
    for source_dataset, df in hf_frames:
        for idx, row in df.iterrows():
            _add_suite_source_record(
                items_by_prompt,
                prompt=row["prompt"],
                source_dataset=source_dataset,
                source_config="default",
                source_split="train",
                source_item_id=idx,
                modality="text",
                category=row.get("category", row.get("source", "")),
                raw_record=_row_dict(row),
            )

    items = list(items_by_prompt.values())
    for item in items:
        item["source_datasets"] = sorted(item["source_datasets"])
        item["num_source_records"] = len(item["source_records"])
        item["modalities"] = sorted(set(record["modality"] for record in item["source_records"]))
        item["categories"] = sorted(set(
            str(record["category"])
            for record in item["source_records"]
            if record.get("category") not in (None, "")
        ))

    metadata = {
        "name": "safetysuite",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "description": "Unified prompt-level safety benchmark dataset. Prompts are deduplicated by leading/trailing whitespace stripped text; all source-specific fields are preserved in source_records.raw_record.",
        "prompt_deduplication": "str(prompt).strip()",
        "num_items": len(items),
        "num_source_records": sum(item["num_source_records"] for item in items),
        "counts_by_source_dataset": {},
    }
    for item in items:
        for record in item["source_records"]:
            source_dataset = record["source_dataset"]
            metadata["counts_by_source_dataset"][source_dataset] = metadata["counts_by_source_dataset"].get(source_dataset, 0) + 1
    metadata["counts_by_source_dataset"] = dict(sorted(metadata["counts_by_source_dataset"].items()))

    return {"metadata": metadata, "items": items}


safetysuite = build_safetysuite()
SAFETYSUITE_PATH = repo_root / "benchmarks" / "safety" / "safetysuite.json"
SAFETYSUITE_PATH.write_text(json.dumps(_jsonable(safetysuite), ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved {SAFETYSUITE_PATH}")
safetysuite["metadata"]
